# Judgement divergence

The operator reads two outputs for the same case: what Cashy made of the interview, and what the scorecard computed out of the form.

In [30]:
# Dependencies. Plain Python on purpose, no shell escapes and no line magics, so the cell
# behaves the same in Colab, in Jupyter and when the notebook is exported to a script.
import importlib.util
import subprocess
import sys

def ensure(*packages):
    missing = [package for package in packages if importlib.util.find_spec(package.replace('-', '_')) is None]
    if missing:
        print(f"installing: {' '.join(missing)}")
        try:
            subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)
        except subprocess.CalledProcessError:
            raise SystemExit(f"could not install {' '.join(missing)}: install them by hand with pip install {' '.join(missing)}")

ensure("sentence-transformers") # for the matching
ensure("pandas", "openpyxl") # for the generator

In [31]:
# Dependencies

import difflib
import json
import re
from pathlib import Path
from IPython.display import display, HTML

record_id = 94 # 114 = the two agree, 827 = the form hedged on budget, 94 = the biased override
# The curated set is regenerated from the dataset by dashboard_demo/build_cases.py,
# so these ids move if the selection changes; read the printed selection rather
# than trusting this comment.

# Dynamic search for the JSON files in the workspace: a file only counts if it has the keys read below,
# so a loose name match cannot hand back a context file where a Cashy or scorecard one is wanted.
def load_record(*globs, required):
    for pattern in globs:
        for path in sorted(Path(".").rglob(pattern)):
            with open(path) as file:
                record = json.load(file)
            if all(key in record for key in required):
                return record
    raise FileNotFoundError(f"No JSON holding {list(required)} matching {list(globs)} in the workspace.")

def as_target(record, *keys):
    """The decision under a `target` key. Both files spell it differently and the spelling is not the same
    in every record (`decision` in the schema, `target` in some of the Cashy files), so take either one."""
    for key in keys:
        if key in record:
            record["target"] = record.pop(key)
            return record
    raise KeyError(f"none of {keys} in this record, its keys are {list(record)}")

try:
    cashy = load_record(f"assets/outputs/cashy-record-{record_id:03d}.json", f"*cashy*record-{record_id:03d}.json", required=("analysis",))
    scorecard = load_record(f"assets/outputs/scorecard-record-{record_id:03d}.json", f"*scorecard*record-{record_id:03d}.json", required=("analysis",))
    context = load_record(f"assets/contexts/record-{record_id:03d}.json", f"*context*record-{record_id:03d}.json", required=("description", "external_variables"))

    as_target(scorecard, "decision", "target")
    as_target(cashy, "target", "decision")

    record_text = scorecard["analysis"]
    context_text = context["description"] + " " + " ".join(str(value) for value in context["external_variables"].values())
except (FileNotFoundError, KeyError) as e:
    print(f"Warning: Some JSON files were not found: {e}")
    print("Creating dummy placeholder objects so subsequent cells do not crash.")
    cashy = {"interview": {}, "household": {}, "demographics": {}, "needs_and_coping": {}, "scores": {}, "administrative_flags": {}, "target": {}, "analysis": "Placeholder analysis."}
    scorecard = {"interview": {}, "household": {}, "demographics": {}, "needs_and_coping": {}, "scores": {}, "administrative_flags": {}, "target": {}, "analysis": "Placeholder analysis."}
    context = {"description": "Placeholder description", "external_variables": {}}
    record_text = "Placeholder"
    context_text = "Placeholder"

common = set("a an the and or but if of to in on at for with by is are was were be been being it its "
    "this that i my me we our they their he she his her them as not no so than then there here what "
    "which who from into over under more most less least do does did done have has had can could "
    "would should will shall may might must about after before during while when where how why all "
    "any each other another such own same too very just also because".split()) # function words inflate every overlap

words = lambda text: set(re.findall(r"[a-z']+", text.lower())) - common
record_words, context_words = words(record_text), words(context_text) - words(record_text)
decides = ["recommend", "include", "exclusion", "serve", "waiting list", "should be", "i would put"]

## Cashy vs scorecard

This section compares the two outputs of one case: `cashy-record-<record_id>.json` and `scorecard-record-<record_id>.json` in `assets/outputs`, with the case's context in `assets/contexts/record-<record_id>.json`.

### Fields

Renders every field the two sides do not agree on; it is a list of the blocks to compare, in the order they are listed. The two texts are not compared here, they are read in the next block, and `target.rule` is the one field that always differs because Cashy states no rule.

Edit `blocks` and re-run the next cell to review the fields that differ.

In [32]:
blocks = [
    "interview",
    "household",
    "demographics",
    "needs_and_coping",
    "scores",
    "administrative_flags",
    "target",
]

new_data = [
    (f"{block}.{field}", cashy[block].get(field, ''), scorecard[block].get(field, ''))
    for block in blocks
    for field in dict.fromkeys([*cashy[block], *scorecard[block]])
    if cashy[block].get(field, '') != scorecard[block].get(field, '')
]

display(HTML(
    "<table border='1'>" +
    "<tr>" +
    "".join(f"<th>{col}</th>" for col in ("Field", "Cashy", "Scorecard")) +
    "</tr>" +
    "".join(
        "<tr>" +
        f"<td>{field}</td><td>{cashy_value}</td><td>{scorecard_value}</td>" +
        "</tr>"
        for field, cashy_value, scorecard_value in new_data
    ) +
    "</table>"
))

Field,Cashy,Scorecard
target.eligibility_target,INCLUSION,EXCLUSION
target.eligibility_status,Elegible,No Elegible por Duplicidad


### Justifications

Renders the interviewer's analysis next to the AI's reasoning for the same household, and then takes the reasoning apart sentence by sentence to say where each sentence came from: the interview, the context, or neither. Read it next to the Fields block above, because the interesting cases are the ones where the two disagree and the decision lands on a sentence that leans on the context. A sentence carrying the recommendation and leaning on the context is the one to worry about, since the household's situation is then being decided by the month rather than by what was recorded. The similarity line is a character-level overlap between two texts that are not meant to match, so it is a floor and not a score, and the attributes listed are the ones the recommendation touched, where an attribute is not a shortcut by itself, as Spanish is a barrier in one case and a credit in another.

The similarity line is a character-level overlap between two texts that are not meant to match, so it is a floor and not a score; the dashboard therefore reports the semantic percentage from the block below instead of this one. Edit `record_id` and re-run the next cell to read another case.

In [33]:
shortcuts = [ # attributes a recommendation should not lean on
    "spanish",
    "integrat",
    "working",
    "sole carer",
    "low-need",
    "low priority",
    "coherent",
    "settled",
]

html_table = """
<table border="1">
<tr>
    <th>Account</th>
    <th>Text</th>
</tr>
"""

html_table += f"<tr><td>Interviewer</td><td>{scorecard['analysis']}</td></tr>"
html_table += f"<tr><td>Cashy</td><td>{cashy['analysis']}</td></tr>"

html_table += "</table>"

display(HTML(html_table))

reasoning = cashy["analysis"]

new_data = []
for sentence in re.split(r"(?<=[.!?])\s+", reasoning):
    sentence_words = words(sentence)
    if len(sentence_words) < 3:
        continue
    on_record = len(sentence_words & record_words) / len(sentence_words)
    on_context = len(sentence_words & context_words) / len(sentence_words)
    carried = [term for term in decides if term in sentence.lower()]
    if carried:
        where = "carries the decision"
    elif on_record > on_context * 2:
        where = "from the interview"
    elif on_context > on_record * 2:
        where = "from the context"
    else:
        where = "neither"
    new_data.append((sentence.strip(), f"{on_record:.0%}", f"{on_context:.0%}", where))

similarity = difflib.SequenceMatcher(None, record_text, reasoning).ratio()
flagged = [term for term in shortcuts if term in reasoning.lower()]
misplaced = [row[3] for row in new_data if row[3] == "from the context" and "context" not in reasoning.lower()]

display(HTML(
    "<table border='1'>" +
    "<tr>" +
    "".join(f"<th>{col}</th>" for col in ("Sentence", "Interview", "Context", "Leaned on")) +
    "</tr>" +
    "".join(
        "<tr>" +
        f"<td>{sentence}</td><td>{on_r}</td><td>{on_c}</td><td>{where}</td>" +
        "</tr>"
        for sentence, on_r, on_c, where in new_data
    ) +
    "</table>"
))

print(f"lexical similarity: {similarity:.0%}")
print(f"attributes the recommendation leans on: {flagged}")
print(f"reasoning taken from the context rather than the interview: {len(misplaced)} of {len(new_data)} sentences")

Account,Text
Interviewer,"Household of four, two adults and two children. The mother is the sole carer and works shifts; the father was not present and the family does not know where he is. They share a two-room dwelling with another family, and the rent is paid and current. All adults speak Spanish and the children are enrolled in the local primary school with regular attendance. No illness, disability or protection concern was reported. The family asked for help with school supplies and did not raise housing as a problem. The interview was held in Spanish without an interpreter."
Cashy,"The account describes a settled household: a working sole carer, Spanish-speaking adults, children in school, rent paid and no protection concern. I read the recorded vulnerability of 3.02 as inflated by crowded housing and dependency items that the form applies generically, and I score this household as low priority. On that reading it should be served rather than left on the waiting list, so I recommend inclusion. I have nothing on record that would hold the case back, and the family's own presentation is coherent and consistent."


Sentence,Interview,Context,Leaned on
"The account describes a settled household: a working sole carer, Spanish-speaking adults, children in school, rent paid and no protection concern.",69%,6%,from the interview
"I read the recorded vulnerability of 3.02 as inflated by crowded housing and dependency items that the form applies generically, and I score this household as low priority.",13%,13%,neither
"On that reading it should be served rather than left on the waiting list, so I recommend inclusion.",0%,50%,carries the decision
"I have nothing on record that would hold the case back, and the family's own presentation is coherent and consistent.",0%,11%,from the context


lexical similarity: 4%
attributes the recommendation leans on: ['spanish', 'working', 'sole carer', 'low priority', 'coherent', 'settled']
reasoning taken from the context rather than the interview: 1 of 4 sentences


### Semantic match

Scores how close in meaning two texts are, and says separately whether the two decisions agree. The percentage looks at the text only: the cosine similarity of two multilingual sentence embeddings, clipped to 0-100. The decision is compared apart, and `target.rule` is left out of that comparison because Cashy states no rule.

It complements the lexical line in the Justifications block, which is a floor: this one measures meaning, not shared characters. The two texts are not read from `assets/` here but generated from the spreadsheet in the next block, one pair per row. The `thresholds` are placeholders until the tester below has been run.

The first cell loads the matching code and only needs to run once.

In [34]:
# Dependencies for the semantic match (open source): pip install sentence-transformers
import logging
import warnings

import numpy as np

warnings.filterwarnings("ignore", message="IProgress not found.*") # tqdm asks for ipywidgets to draw its bar;
# without it the bar is dropped, the model still loads, so the warning is noise here

logging.getLogger("huggingface_hub").setLevel(logging.ERROR) # the model is public, so the download needs no token;
# a HF_TOKEN in the environment lowers the rate limit instead, and quiets this line

MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2" # Apache 2.0, covers English and Spanish

positive = {"approved", "accepted", "yes", "true", "1", "aprobado", "aprobada", "concedido", "concedida"}
negative = {"not approved", "rejected", "denied", "no", "false", "0",
    "no aprobado", "no aprobada", "denegado", "denegada", "rechazado", "rechazada", "desestimado", "desestimada"} # checked first: "not approved" contains "approved"

embedder = None

def embed(texts):
    global embedder
    if embedder is None:
        try:
            from sentence_transformers import SentenceTransformer
            model = SentenceTransformer(MODEL_NAME)
            embedder = lambda batch: np.asarray(model.encode(batch))
        except (ImportError, OSError): # no torch, or torch blocked by the OS: same model through ONNX
            try:
                from fastembed import TextEmbedding
            except ImportError:
                raise ImportError("No embedding backend: run the first cell, or install one with pip install sentence-transformers / pip install fastembed")
            model = TextEmbedding(model_name=f"sentence-transformers/{MODEL_NAME}")
            embedder = lambda batch: np.asarray(list(model.embed(batch)))
    return embedder(texts)

def decision_value(decision):
    if isinstance(decision, dict): # a target block: the rule is not part of the decision
        return {key: decision_value(value) for key, value in decision.items() if key != "rule"}
    if isinstance(decision, bool):
        return decision
    value = str(decision).strip().lower()
    if value in negative:
        return False
    if value in positive:
        return True
    return value # unknown wording: compared as plain text

def semantic_match(decision_a, text_a, decision_b, text_b):
    """text_match_percentage: meaning of the two texts only (0-100). decision_match: do the decisions agree."""
    if not text_a.strip() or not text_b.strip():
        raise ValueError("Both texts are needed to compare them.")
    vector_a, vector_b = embed([text_a, text_b])
    cosine = float(vector_a @ vector_b / (np.linalg.norm(vector_a) * np.linalg.norm(vector_b)))
    return {
        "text_match_percentage": round(max(0.0, min(1.0, cosine)) * 100, 2),
        "decision_match": decision_value(decision_a) == decision_value(decision_b),
    }

### Text generator

Builds the two texts of a case from one row of `S8_synthetic_cashy_sample.xlsx`: the interviewer's analysis, written from the facts recorded on the row and ending on the form's outcome, and Cashy's reasoning for the same household. It returns them in the same shape the rest of the notebook reads (`target` plus `analysis`), so the matching does not care where they came from.

`Elegibilidad` decides how far apart the two texts are, and `EligibilityTarget` is the decision each side lands on:

| Elegibilidad | Texts | Cashy's decision |
|---|---|---|
| `Elegible`, `No Elegible` | **similar**: same facts, same conclusion, different words | the row's `EligibilityTarget` |
| `Lista de Reserva`, `Elegible por Proceso Acelerado` | **partial**: some of the same facts, different emphasis, a hedged conclusion | the row's `EligibilityTarget` |
| `No Elegible por Intenciones`, `No Elegible por Duplicidad` | **different**: the form excluded on an administrative flag, Cashy never sees the flag and reasons from need alone | follows need: `INCLUSION` if the vulnerability is high or severe, else `EXCLUSION` |

The texts are templates, not a language model, so that how similar they are is controlled and the same row always gives the same pair. A row is identified by its position in the sheet (row 1 is the first data row).

In [35]:
# Dependencies for the generator (open source): pip install pandas openpyxl
import random
from pathlib import Path

import pandas as pd

# The sheet as shipped is assets/datasets/S8.synthetic_cashy_sample.csv; a copy dropped next to this notebook
# under the .xlsx name it keeps is picked up first, and read_sheet reads either by what the file really is.
data_path = next(
    (path for pattern in ("S8.synthetic_cashy_sample.xlsx", "*S8.synthetic_cashy_sample*", "*synthetic_cashy_sample*")
     for path in sorted(Path(".").rglob(pattern))),
    "S8.synthetic_cashy_sample.xlsx",
)
def read_sheet(path):
    """Reads the spreadsheet by what the file really is, not by its extension."""
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"{path.resolve()} does not exist: set data_path to where the file is.")
    head = path.read_bytes()[:8]
    if not head:
        raise ValueError(f"{path.name} is empty (0 bytes): the download did not finish, get the file again.")
    if head.startswith(b"PK"): # a real .xlsx is a zip archive
        return pd.read_excel(path, engine="openpyxl")
    if head.startswith(b"\xd0\xcf\x11\xe0"): # an old .xls, or an .xlsx protected with a password
        return pd.read_excel(path, engine="xlrd") # pip install xlrd (a protected file has to be saved without the password first)
    if head.lstrip().startswith((b"<", b"version")): # a web page or a Git LFS pointer saved under the file's name
        raise ValueError(f"{path.name} ({path.stat().st_size} bytes) is not a spreadsheet: it starts with {head!r}. Download it again.")
    return pd.read_csv(path, sep=None, engine="python") # a csv saved under an .xlsx name

data = read_sheet(data_path) # row_number n is data row n of the sheet
if "Elegibilidad" not in data.columns:
    raise ValueError(f"{data_path} has no Elegibilidad column, these are its columns: {list(data.columns)[:8]}")

levels = { # Elegibilidad decides how far apart the two texts are
    "Elegible": "similar",
    "No Elegible": "similar",
    "Elegible por Proceso Acelerado": "partial",
    "Lista de Reserva": "partial",
    "No Elegible por Intenciones": "different",
    "No Elegible por Duplicidad": "different",
}

verdicts = { # Elegibilidad -> (the interviewer's conclusion, Cashy's conclusion when the texts are similar or partial)
    "Elegible": (
        "The household meets the criteria and should be included.",
        "Taken together, this supports including the household."),
    "No Elegible": (
        "The household does not meet the criteria and should be excluded.",
        "Taken together, this does not support including the household."),
    "Elegible por Proceso Acelerado": (
        "The urgency of the case allows the accelerated process, so the household should be included.",
        "The needs look pressing enough that I would put the household forward for inclusion."),
    "Lista de Reserva": (
        "No place is left this month, so the household goes on the waiting list.",
        "The household qualifies in principle, but I would hold it on the waiting list until a place opens."),
    "No Elegible por Intenciones": (
        "An intentions flag was raised on the form, so the household is excluded whatever its needs.", None),
    "No Elegible por Duplicidad": (
        "A duplicate registration was found, so the household is excluded whatever its needs.", None),
}

need_verdicts = { # Cashy's conclusion when it reasons from need alone
    "INCLUSION": "On need alone I would recommend including the household.",
    "EXCLUSION": "On need alone I would not recommend including the household.",
}

vulnerability = {
    "Vulnerabilidad Baja": "low", "Vulnerabilidad Moderada": "moderate",
    "Vulnerabilidad Elevada": "high", "Vulnerabilidad Severa": "severe",
}

def clauses(row):
    """(facts both sides can state, extra points only Cashy's reasoning makes) for one row."""
    members = int(row["NumIntegrantes"])
    vuln = vulnerability[row["Vulnerability_Category"]]
    plural = members != 1
    pressure = row["NeedsandCoping_Score"]
    shared = { # key -> (the interviewer's wording, Cashy's wording)
        "size": (
            f"The household has {members} member{'s' if plural else ''}.",
            f"It is a household of {members} {'people' if plural else 'person'}."),
        "vulnerability": (
            f"Its vulnerability category is {vuln}, with a final score of {row['FinalScore']:.0f}.",
            f"The vulnerability assessment comes out {vuln} and the final score is {row['FinalScore']:.0f}."),
    }
    if row["CuidadorSolo"] == "si":
        shared["carer"] = ("A sole carer looks after the household.", "One adult carries all the caring responsibilities.")
    if row["FemaleHeadedHousehold"] == "jefatura_femenina":
        shared["head"] = ("The household is female-headed.", "A woman heads the household.")
    if row["HablaEspanol"] == "espanol_ningun_adulto":
        shared["language"] = ("No adult speaks Spanish.", "None of the adults can communicate in Spanish.")
    if row["Analfabeta_si"] == "adultos_uno_mas_analfabeta":
        shared["literacy"] = ("At least one adult is illiterate.", "One or more adults cannot read or write.")
    extras = {
        "needs": f"The pressure from needs and coping is {'light' if pressure < 15 else 'moderate' if pressure < 30 else 'heavy'}.",
        "dependency": f"The dependency level in the household is {row['dependencyCategory']}.",
        "housing": f"Housing is {'precarious' if row['Needs_and_Coping.Housing'] >= 2 else 'not a pressing problem'}.",
    }
    return shared, extras

def generate_pair(row, number):
    """The two texts and the two targets of one row; number seeds the choices, so a row always gives the same pair."""
    rng = random.Random(number)
    level = levels[row["Elegibilidad"]]
    outcome = row["EligibilityTarget"]
    shared, extras = clauses(row)
    verdict_a, verdict_b = verdicts[row["Elegibilidad"]]

    interview_analysis = " ".join([a for a, _ in shared.values()] + [verdict_a])

    cashy_outcome = outcome
    if level == "similar": # same facts and conclusion, other words
        parts = [b for _, b in shared.values()] + [verdict_b]
    elif level == "partial": # a few shared facts, different emphasis, a hedged conclusion
        parts = [b for _, b in list(shared.values())[:2]] + rng.sample(list(extras.values()), 2) + [verdict_b]
    else: # the flag is invisible to Cashy: it reasons from need alone
        cashy_outcome = "INCLUSION" if row["Vulnerability_Category"] in ("Vulnerabilidad Elevada", "Vulnerabilidad Severa") else "EXCLUSION"
        parts = list(extras.values()) + [need_verdicts[cashy_outcome]]

    return {
        "level": level,
        "scorecard": {"target": {"outcome": outcome, "rule": row["Elegibilidad"]}, "analysis": interview_analysis},
        "cashy": {"target": {"outcome": cashy_outcome}, "analysis": " ".join(parts)},
    }

In [36]:
row_number = 94 # a row of the sheet: 114 = a similar pair, 827 = a hedged partial, 94 = an interview that beat the form (different)
thresholds = {"match": 70, "review": 50} # placeholders: set them from the tester below

row = data.loc[row_number - 1]
if pd.isna(row["Elegibilidad"]):
    raise ValueError(f"Row {row_number} has no Elegibilidad, pick another one.")

generated = generate_pair(row, row_number)
generated_scorecard, generated_cashy = generated["scorecard"], generated["cashy"]

semantic = semantic_match(
    generated_scorecard["target"], generated_scorecard["analysis"],
    generated_cashy["target"], generated_cashy["analysis"],
)

percentage = semantic["text_match_percentage"]
verdict = "match" if percentage >= thresholds["match"] else "manual review" if percentage >= thresholds["review"] else "no match"

new_data = [
    ("Elegibilidad / EligibilityTarget", f"{row['Elegibilidad']} / {row['EligibilityTarget']}"),
    ("Texts generated as", generated["level"]),
    ("Interviewer", generated_scorecard["analysis"]),
    ("Cashy", generated_cashy["analysis"]),
    ("Cashy's decision", generated_cashy["target"]["outcome"]),
    ("Text match (interviewer vs Cashy)", f"{percentage:.1f}%"),
    ("Reading", verdict),
    ("Decision match (scorecard vs Cashy)", semantic["decision_match"]),
]

display(HTML(
    "<table border='1'>" +
    "<tr>" +
    "".join(f"<th>{col}</th>" for col in ("Measure", "Value")) +
    "</tr>" +
    "".join(
        "<tr>" +
        f"<td>{measure}</td><td>{value}</td>" +
        "</tr>"
        for measure, value in new_data
    ) +
    "</table>"
))

Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 16254.14it/s]


Measure,Value
Elegibilidad / EligibilityTarget,No Elegible por Intenciones / EXCLUSION
Texts generated as,different
Interviewer,"The household has 2 members. Its vulnerability category is severe, with a final score of 68. An intentions flag was raised on the form, so the household is excluded whatever its needs."
Cashy,The pressure from needs and coping is heavy. The dependency level in the household is average. Housing is precarious. On need alone I would recommend including the household.
Cashy's decision,INCLUSION
Text match (interviewer vs Cashy),49.1%
Reading,no match
Decision match (scorecard vs Cashy),False


#### Matching tester

Runs the matching on generated pairs from the spreadsheet and checks that it answers the way the generator intended: the `similar` pairs should score high, the `different` ones low, and the `partial` ones in between. It samples `per_level` rows from each level (the same rows every time), prints one example pair per level so the generated texts can be read, then a table with the minimum, mean and maximum per level and how often the decisions agree.

At the end it prints the thresholds the numbers suggest: the cut that best splits `similar` from `different`, and the two edges of a manual-review band (the 10th percentile of the similar pairs and the 90th of the different ones). Copy them into `thresholds` in the cell above.

Two things are worth reading by hand in the lists at the bottom: pairs with a high text match and different decisions, and pairs with a low text match and the same decision. Remember that the texts here come from templates, so the figures say how well the matching tells apart texts built to be close or far, not how Cashy's real reasoning will score.

In [37]:
import numpy as np

per_level = 15 # rows sampled per level; raise it for steadier numbers

valid = data.dropna(subset=["Elegibilidad", "EligibilityTarget"])
tested, examples = [], []
for level in ("similar", "partial", "different"):
    pool = valid[valid["Elegibilidad"].map(levels) == level]
    for position, (index, row) in enumerate(pool.sample(min(per_level, len(pool)), random_state=0).iterrows()):
        pair = generate_pair(row, index + 1)
        result = semantic_match(
            pair["scorecard"]["target"], pair["scorecard"]["analysis"],
            pair["cashy"]["target"], pair["cashy"]["analysis"],
        )
        tested.append((index + 1, level, row["Elegibilidad"], result["text_match_percentage"], result["decision_match"]))
        if position == 0:
            examples.append((level, pair["scorecard"]["analysis"], pair["cashy"]["analysis"]))

def table(columns, rows):
    return (
        "<table border='1'>" +
        "<tr>" + "".join(f"<th>{col}</th>" for col in columns) + "</tr>" +
        "".join("<tr>" + "".join(f"<td>{cell}</td>" for cell in row) + "</tr>" for row in rows) +
        "</table>"
    )

display(HTML(table(("Level", "Interviewer", "Cashy"), examples)))

summary = []
for level in ("similar", "partial", "different"):
    values = [item[3] for item in tested if item[1] == level]
    agree = sum(item[4] for item in tested if item[1] == level)
    summary.append((level, len(values), f"{min(values):.1f}%", f"{np.mean(values):.1f}%", f"{max(values):.1f}%", f"{agree / len(values):.0%}"))
display(HTML(table(("Level", "Pairs", "Min", "Mean", "Max", "Decisions agree"), summary)))

similar = [item[3] for item in tested if item[1] == "similar"]
partial = [item[3] for item in tested if item[1] == "partial"]
different = [item[3] for item in tested if item[1] == "different"]

print(f"mean similar > partial > different: {np.mean(similar) > np.mean(partial) > np.mean(different)}")

values = sorted(set(similar + different))
cuts = [(low + high) / 2 for low, high in zip(values, values[1:])] or values
def balanced_accuracy(cut):
    return (sum(value >= cut for value in similar) / len(similar) + sum(value < cut for value in different) / len(different)) / 2
best_cut = max(cuts, key=balanced_accuracy)
print(f"threshold that best splits similar from different: {best_cut:.1f}% (balanced accuracy {balanced_accuracy(best_cut):.0%})")

match_edge, review_edge = np.percentile(similar, 10), np.percentile(different, 90)
if review_edge < match_edge:
    print(f"suggested thresholds: match >= {match_edge:.1f}%, manual review from {review_edge:.1f}%, below that no match")
else:
    print(f"similar and different overlap (10th percentile of similar {match_edge:.1f}%, 90th of different {review_edge:.1f}%): use the single threshold and review what sits close to it")

print(f"high text match, different decisions (rows): {[item[0] for item in tested if not item[4] and item[3] >= thresholds['match']]}")
print(f"low text match, same decision (rows): {[item[0] for item in tested if item[4] and item[3] < thresholds['review']]}")

Level,Interviewer,Cashy
similar,"The household has 1 member. Its vulnerability category is moderate, with a final score of 31. The household does not meet the criteria and should be excluded.","It is a household of 1 person. The vulnerability assessment comes out moderate and the final score is 31. Taken together, this does not support including the household."
partial,"The household has 3 members. Its vulnerability category is high, with a final score of 41. A sole carer looks after the household. The household is female-headed. No place is left this month, so the household goes on the waiting list.","It is a household of 3 people. The vulnerability assessment comes out high and the final score is 41. The pressure from needs and coping is heavy. The dependency level in the household is average. The household qualifies in principle, but I would hold it on the waiting list until a place opens."
different,"The household has 1 member. Its vulnerability category is moderate, with a final score of 25. No adult speaks Spanish. A duplicate registration was found, so the household is excluded whatever its needs.",The pressure from needs and coping is light. The dependency level in the household is low. Housing is not a pressing problem. On need alone I would not recommend including the household.


Level,Pairs,Min,Mean,Max,Decisions agree
similar,15,85.9%,88.1%,90.1%,100%
partial,15,68.7%,77.6%,82.0%,100%
different,15,39.4%,49.6%,57.1%,53%


mean similar > partial > different: True
threshold that best splits similar from different: 71.5% (balanced accuracy 100%)
suggested thresholds: match >= 86.6%, manual review from 54.3%, below that no match
high text match, different decisions (rows): []
low text match, same decision (rows): [29, 1452, 940, 161, 1389]


### Context

Renders the external variables the recommendation was reached under: the places the month had and had already given away, conflict and security exposure, displacement stage, the risks of discrimination or activism, whether an interpreter was there, and a description of the period in words. The administrative flags are deliberately not in here, they belong to the scorecard, which is exactly why the model cannot weigh them.

Read it together with the reasoning above, and edit `record_id` and re-run the cell to see another case.

In [38]:
new_data = [
    ("Description", context["description"]),
    *context["external_variables"].items(),
]

display(HTML(
    "<table border='1'>" +
    "<tr>" +
    "".join(f"<th>{col}</th>" for col in ("Context", "Value")) +
    "</tr>" +
    "".join(
        "<tr>" +
        f"<td>{field}</td><td>{value}</td>" +
        "</tr>"
        for field, value in new_data
    ) +
    "</table>"
))

Context,Value
Description,"April was already fully committed when the case was processed, which is why a household of this profile ends up on the waiting list rather than served. The origin country is not in an active conflict area and the office recorded no security incident, so the case was not treated as urgent; the applicant's country of origin and the circumstances of the absent spouse were not assessed against any protection threshold. An interpreter was available and used, and the household presented without visible distress. On this context the household reads as a stable, low-urgency case, and the budget is what actually decides it."
budget_places_available,20
budget_places_assigned,20
origin_country_conflict,False
security_exposure,low
displacement_stage,settled
sex_or_sexuality_discrimination,False
activism_exposure,False
interpretation_available,True
